In [1]:
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import os
from sentence_transformers import SentenceTransformer

In [3]:
load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=api_key)

In [4]:
index_name = "pinecone-tut"
dimension = 384

if index_name not in [index['name'] for index in pc.list_indexes()]:
    pc.create_index(
        name = index_name,
        dimension= dimension,
        metric='cosine',
        spec = ServerlessSpec(cloud='aws', region='us-east-1')
    )
    print(f"Created Index: {index_name}")

Created Index: pinecone-tut


In [5]:
index = pc.Index(index_name)
print(f"Index stats: {index.describe_index_stats()}")

Index stats: DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


In [6]:
documents = [
    "The quick borwn fox jumps over the lazy dog",
    "AI is transforming technology",
    "Python is a popular programming language",
    "ML models require large datasets",
    "Vector databases enable fast similarity search",
    "NLP analyzes text",
    "DL uses neural networks",
    "Data Science combines statistics and programming",
    "Software development involves writing code"
]

model = SentenceTransformer('all-miniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1517.28it/s]


In [7]:
embeddings = model.encode(documents).tolist()

to_upsert = [
    {
        "id": f"doc{i}",
        "values": embeddings[i],
        "metadata": {"text": documents[i]}
    }
    for i in range(len(documents))
]

In [8]:
index.upsert(vectors=to_upsert)
print("Documents inserted sucessfully.")

Documents inserted sucessfully.


In [9]:
print(f"Index stats: {index.describe_index_stats()}")

Index stats: DescribeIndexStatsResponse(dimension=384, total_vector_count=9, metric='cosine', namespaces=1)


In [10]:
query = "What is AI and Machine Learning?"

query_embedding = model.encode(query)

search_results = index.query(
    vector = query_embedding.tolist(),
    top_k=3,
    include_metadata=True
)

for i, match in enumerate(search_results['matches']):
    score = match['score']
    text = match['metadata']['text']
    
    print(f"{i+1}. Similarity score: {score:.3f}")
    print(f"Text: {text}")
    print()

1. Similarity score: 0.544
Text: AI is transforming technology

2. Similarity score: 0.365
Text: Data Science combines statistics and programming

3. Similarity score: 0.339
Text: DL uses neural networks

